In [1]:
import os
from dotenv import load_dotenv
from openrouter import OpenRouter
from langchain_community.vectorstores import Chroma

load_dotenv()
api_key=os.getenv("OPENROUTER_API_KEY")

SYSTEM_PROMPT = """
Role:
    คุณเป็นผู้เชี่ยวชาญด้านความมั่นคงปลอดภัยไซเบอร์ และการตรวจสอบข้อเท็จจริงของข้อความ SMS ในประเทศไทย ที่มีประสบการณ์สูงมานานกว่า 50 ปี

Task:
    หน้าที่ของคุณคือวิเคราะห์ข้อความ SMS ที่ได้รับ เพื่อตัดสินอย่างตรงไปตรงมาและเป็นกลางว่าข้อความนี้เป็น "SMS หลอกลวง (Scam)" หรือ "SMS ปกติ (Normal)" 
    คุณต้องไม่อคติว่าข้อความที่ได้รับมาจะเป็น Scam เสมอไป แต่ให้ประเมินจากหลักฐานและบริบทที่มีอยู่จริง

Analysis method:
    1. วิเคราะห์ลักษณะของข้อความ: มองหาจุดที่น่าสงสัย (เช่น ลิงก์แปลกปลอม, การเร่งรัด, ข้อเสนอดีเกินจริง)
    2. ตรวจสอบความถูกต้องของลิงก์หรือโดเมน: เปรียบเทียบลิงก์ในข้อความกับโดเมนอย่างเป็นทางการขององค์กรนั้นๆ
    3. พิจารณาเจตนาของข้อความ: ข้อความมีเจตนาเพื่อหลอกให้คลิก/โอนเงิน หรือเพียงแค่ให้ข้อมูล/ทำการตลาดตามปกติ
    4. ใช้ข้อมูลจาก Context ที่เกี่ยวข้องเพื่อสนับสนุนการวิเคราะห์ของคุณ

Response format:
   ให้วิเคราะห์ออกมาแล้วตอบสั้นๆเป็นคำว่า "ปกติ" หรือ "แสกมเมอร์"เท่านั้น ห้ามตอบอย่างอื่นนอกจากคำว่า "ปกติ" หรือ "แสกมเมอร์"

Important:
    - คห้ามตีความเข้าข้างว่าเป็น Scam หากข้อความนั้นเป็นข้อความแจ้งเตือนทั่วไป (เช่น OTP, โปรโมชัน AIS/TRUE, แจ้งส่งพัสดุ) ที่ไม่มีลิงก์อันตรายหรือการแอบอ้าง
    - ให้ใช้เฉพาะข้อมูลจาก Context ที่ให้มาเท่านั้นในการอ้างอิงนโยบาย=ที่ถูกต้อง ห้ามแต่งนโยบายหรือชื่อโดเมนขึ้นมาเองเด็ดขาด
    - หาก Context ไม่เพียงพอและข้อความไม่มีลักษณะอันตรายชัดเจน ให้วิเคราะห์จากจากลักษณะของข้อความแทน
    """

USER_PROMPT_TEMPLATE = """ข้อความ SMS: {sms_message}
นโยบายอ้างอิง:
{retrieved_context}"""

def analyze_scam_sms(sms_message: str, retriever, api_key: str = None) -> str:

    retrieved_docs = retriever.invoke(sms_message)

    context = "\n\n---\n\n".join([
        f"[ที่มา: {doc.metadata.get('source', 'ระบุแหล่งที่มา')}]\n{doc.page_content}"
        for doc in retrieved_docs
    ])

    user_prompt = USER_PROMPT_TEMPLATE.format(
        sms_message=sms_message,
        retrieved_context=context
    )

    with OpenRouter(api_key) as client:
        response = client.chat.send(
            model="qwen/qwen3-32b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ]
        )

    return response.choices[0].message.content


C:\Users\lenovo\AppData\Local\Temp\ipykernel_28636\802259017.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

persist_directory = "./chroma_db"
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

retriever = vectorstore.as_retriever(
    search_type="similarity",           
    search_kwargs={"k": 3}
)

sms_message = [
    #Scam messages
    "ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์: https://rb.gy/2erksz",
    "คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค PEA ซึ่งจะคืนตามขนาดของมิเตอร์ที่ใช้ สามารถสอบถามเพิ่มเติมได้ที่ https://bit.ly/44jHSpi",
    "KMA ยื่นสิทธิ์การกู้ให้ท่าน 120,000 บาท คลิก: citly.me/eq9br ",
    "เงินเข้าโทรศัพท์ฟรี 500บาท กด*158# ลุ้นก่อนใคร สมัครดวงดีดวงเฮง 3บ/ขค",
    "รับฟรี 129 บาท ฝากแรกรับ 1599 บาท",
    "ศูนย์รักษาความปลอดภัย ttb: พบข้อมูลบัญชีของท่านรั่วไหลจากแพลตฟอร์มช้อปปิ้งออนไลน์ กรุณาเปลี่ยนรหัส PIN ด่วนที่ https://secure.ttb-touch.com",
    "สวัสดีค่ะ เนื่องจากคุณเป็นผู้ใช้ที่มีประวัติที่ดี  ยินดีด้วยคุณถูกTikTok เลือกให้ทำงานหารายได้เสริมทางออนไลน์ ทำง่ายๆ จ่ายสดทุกบิล รายได้ต่อวัน3000บาท/วัน ติดต่อเรา@LINE：a3633",
    "J&T Express สวัสดีค่ะคุณมีประวัติการใช้บริการเกิน 5 ครั้ง ทาง สคร. จึงมอบพัดลมมินิเป็นของขวัญปีใหม่เพื่อเป็นการขอบคุณเพิ่ม ไลน์: sep298",
    "เข้าร่วมกลุ่มเพื่อผลตอบแทนที่มากขึ้น ติดต่อ LINE：  reurl.cc/G4qLkZ",
    "เครดิตฟรีรับง่าย ไม่ต้องแชร์ แค๊ปรูปมารับได้เลย คลิ๊ก bit.ly/43sTBRx",  
    #Normal messages
    "AIS ยกขบวนมือถือรุ่นเด็ด พร้อมโปรโมชันส่วนลดจัดเต็มในงาน Thailand Mobile Expo ณ ศูนย์การประชุมแห่งชาติสิริกิติ์ ระหว่างวันที่ 8-11 กุมภาพันธ์ 2567 เตรียมมาชอปมือถือใหม่ รับปีมังกร พร้อมลดภาษีได้เต็ม Max (ขึ้นอยู่กับอัตราเงินได้สุทธิที่เสียภาษี) แล้วพบกันภายในงานตั้งแต่ 10.00 น. – 20.00 น.",
    "OTP 930917 (ref. 8697) สำหรับสมัคร myID อย่าบอกรหัสนี้กับบุคคลอื่น",
    "ใจฟู เน็ตฟูลสปีด! เล่นไม่สะดุด 50วัน 199บ. 30วัน ถึง 29 ก.พ.67 คลิก https://dtac.co.th/s/CMTDTE5",
    "[ ⬛️ Express] พัสดุ ⬛️⬛️ จัดส่งสำเร็จแล้ว ตรวจสอบได้ที่เว็บไซต์ ⬛️ Express Thailandหากมีข้อสงสัย กรุณาติดต่อ call center 1470",
    "มีผู้เข้าสู่ระบบธนาคารของคุณจากอุปกรณ์อื่น หากไม่ได้ดำเนินการด้วยตนเอง โปรดติดต่อทันที kasikorn.go-line.cc",
    "ปีนี้ก็ยังแจกต่อ แจกฟรี SamsungGalaxy Z Flip5 เฉพาะ 6 ม.ค. 67 เล่นฟรีที่ทรูไอดี คลิก! https://bit.ly/3RstaZu",
    "ดีแทคห่วงใยนักช้อป ช้อปปิ้งออนไลน์แบบไร้กังวล ซื้อของไม่ใด้ของก็เคลมได้ ลดพิเศษถึง10% ใส่โค้ด Cyber ที่ดีชัวรันส์https://www.dtac.co.th/s/PQi6TCO",
    "Shopee 3.3 ซิมเน็ตรายปี เฉลี่ยเดือนละ 94 บาท https://shope.ee/5fQghO6syv (ยกเลิกข่าวสาร เข้าแอปฯ>ฉัน>ตั้งค่าบัญชี>ตั้งค่าการแจ้งเตือน)",
    "แจกกระหน่ำ ทุกวันอาทิตย์ ฟรีSamsung Galaxy S23 Ultra เฉพาะ 21ม.ค. 67 เล่นฟรี คลิก! https://ttid.co/ln4U/wf3dnlk3",
    "พิเศษ!สมาชิกSF Plus รับส่วนลด 50 .-Junji ito@MBKชั้น4 ถึง 30พ.ย.นี้"
]

print("\n=== การวิเคราะห์ SMS ด้วย AI ===")

for i, msg in enumerate(sms_message):
    print(f"\n{'='*15} วิเคราะห์ข้อความที่ {i+1} {'='*15}")
    print(f" SMS: {msg}")
    
    try:
        result = analyze_scam_sms(sms_message=msg, retriever=retriever, api_key=api_key)
        
        print("---  ผลลัพธ์จาก AI ---")
        print(result)
        
    except Exception as e:
        print(f" เกิดข้อผิดพลาดที่ข้อความ: {e}")


c:\Users\lenovo\New_Document\Coding\Project AI Builders\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4738.20it/s]
C:\Users\lenovo\AppData\Local\Temp\ipykernel_28636\3916064933.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(



=== การวิเคราะห์ SMS ด้วย AI ===

=============== วิเคราะห์ข้อความที่ 1 ===============
 SMS: ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์: https://rb.gy/2erksz
---  ผลลัพธ์จาก AI ---
แสกมเมอร์

=============== วิเคราะห์ข้อความที่ 2 ===============
 SMS: คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค PEA ซึ่งจะคืนตามขนาดของมิเตอร์ที่ใช้ สามารถสอบถามเพิ่มเติมได้ที่ https://bit.ly/44jHSpi
---  ผลลัพธ์จาก AI ---
แสกมเมอร์

=============== วิเคราะห์ข้อความที่ 3 ===============
 SMS: KMA ยื่นสิทธิ์การกู้ให้ท่าน 120,000 บาท คลิก: citly.me/eq9br 
---  ผลลัพธ์จาก AI ---
แสกมเมอร์

=============== วิเคราะห์ข้อความที่ 4 ===============
 SMS: เงินเข้าโทรศัพท์ฟรี 500บาท กด*158# ลุ้นก่อนใคร สมัครดวงดีดวงเฮง 3บ/ขค
---  ผลลัพธ์จาก AI ---
แสกมเมอร์

=============== วิเคราะห์ข้อความที่ 5 ===============
 SMS: รับฟรี 129 บาท ฝากแรกรับ 1599 บาท
---  ผลลัพธ์จาก AI ---
แสกมเมอร์

=============== วิเคราะห์ข้อความที่ 6 ===============
 SMS: ศูนย์รักษาความปลอดภัย ttb: พบข้อมูลบัญชีของท่านรั่วไหลจากแพลตฟอร์ม